In [2]:
import os
from dotenv import load_dotenv
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

load_dotenv()

client = QdrantClient(
    url=os.getenv("QDRANT_URL"),
    api_key=os.getenv("QDRANT_API_KEY"),
)

print(client.get_collections())

collections=[CollectionDescription(name='test')]


In [ ]:
# Insset new data into the collection
client.create_collection("test", vectors_config=VectorParams(size=4, distance=Distance.COSINE))
client.upsert("test", points=[
    PointStruct(id=1, vector=[0.1, 0.2, 0.3, 0.4], payload={"text": "hello"}),
    PointStruct(id=2, vector=[0.9, 0.1, 0.0, 0.2], payload={"text": "world"}),
])
print(client.query_points("test", query=[0.1, 0.2, 0.3, 0.5], limit=1).points)

[ScoredPoint(id=1, version=1, score=0.9939991, payload={'text': 'hello'}, vector=None, shard_key=None, order_value=None)]


In [ ]:
# Update existing data in the collection
client.upsert("test", points=[
    PointStruct(id=2, vector=[0.4, 0.3, 0.2, 0.1], payload={"text": "qdrant"}),
])
print(client.retrieve("test", ids=[2], with_vectors=True))

[Record(id=2, payload={'text': 'qdrant'}, vector=[0.7302968, 0.54772264, 0.3651484, 0.1825742], shard_key=None, order_value=None)]


In [4]:
# Add 18 more points (ids 3-20) with random vectors
import random

random.seed(42)
words = [
    "apple", "banana", "cherry", "dog", "cat", "horse",
    "python", "rust", "java", "red", "green", "blue",
    "sun", "moon", "star", "river", "mountain", "ocean",
]

client.upsert("test", points=[
    PointStruct(
        id=i,
        vector=[round(random.random(), 2) for _ in range(4)],
        payload={"text": word},
    )
    for i, word in enumerate(words, start=3)
])
print(client.count("test"))

count=20


In [5]:
# SQL-like search: filter on payload only, no vector similarity
from qdrant_client.models import Filter, FieldCondition, MatchValue, MatchAny, PayloadSchemaType

# Index the field we filter on (Qdrant Cloud may reject filters on unindexed fields)
client.create_payload_index("test", field_name="text", field_schema=PayloadSchemaType.KEYWORD)

# SELECT * FROM test WHERE text = 'dog'
points, _ = client.scroll(
    "test",
    scroll_filter=Filter(must=[FieldCondition(key="text", match=MatchValue(value="dog"))]),
)
print(points)

# SELECT * FROM test WHERE text IN ('red', 'green', 'blue')
points, _ = client.scroll(
    "test",
    scroll_filter=Filter(must=[FieldCondition(key="text", match=MatchAny(any=["red", "green", "blue"]))]),
)
print([(p.id, p.payload["text"]) for p in points])

# SELECT COUNT(*) FROM test WHERE text NOT IN ('red', 'green', 'blue')
print(client.count(
    "test",
    count_filter=Filter(must_not=[FieldCondition(key="text", match=MatchAny(any=["red", "green", "blue"]))]),
))

[Record(id=6, payload={'text': 'dog'}, vector=None, shard_key=None, order_value=None)]
[(12, 'red'), (13, 'green'), (14, 'blue')]
count=17


In [8]:
# Hybrid search (dense vector + BM25)
# A sparse vector can't be added to an existing collection, so recreate "test" with both
import random
from qdrant_client.models import SparseVectorParams, Modifier, Document, Prefetch, FusionQuery, Fusion

client.delete_collection("test")
client.create_collection(
    "test",
    vectors_config=VectorParams(size=4, distance=Distance.COSINE),
    sparse_vectors_config={"bm25": SparseVectorParams(modifier=Modifier.IDF)},
)

random.seed(42)
all_words = ["hello", "qdrant"] + words
client.upsert("test", points=[
    PointStruct(
        id=i,
        vector={
            "": [round(random.random(), 2) for _ in range(4)],      # dense (the unnamed vector)
            "bm25": Document(text=word, model="qdrant/bm25"),     # BM25, computed by Qdrant from the text
        },
        payload={"text": word},
    )
    for i, word in enumerate(all_words, start=1)
])

# Run both searches, then merge the two rankings with RRF
result = client.query_points(
    "test",
    prefetch=[
        Prefetch(query=[0.1, 0.2, 0.3, 0.5], limit=5),                                 # dense
        Prefetch(query=Document(text="dog", model="qdrant/bm25"), using="bm25", limit=5),  # BM25
    ],
    query=FusionQuery(fusion=Fusion.RRF),
    limit=5,
    with_vectors=True,  # vectors are not returned by default
)
for p in result.points:
    assert isinstance(p.vector, dict)  # named vectors come back as a dict; also tells the type checker
    print(p.id, round(p.score, 3), p.payload["text"])
    print("   dense:", p.vector[""])
    print("   bm25: ", p.vector["bm25"])

6 0.5 dog
   dense: [0.7139128, 0.61696166, 0.2996671, 0.1410198]
   bm25:  indices=[1312749093] values=[1.6877434]
18 0.5 river
   dense: [0.3046928, 0.25708455, 0.19995464, 0.8950351]
   bm25:  indices=[12344922] values=[1.6877434]
4 0.333 banana
   dense: [0.03452612, 0.23017414, 0.74806595, 0.6214702]
   bm25:  indices=[178115665] values=[1.6877434]
12 0.25 red
   dense: [0.20816116, 0.59846336, 0.26020145, 0.7285641]
   bm25:  indices=[347068675] values=[1.6877434]
14 0.2 blue
   dense: [0.20403671, 0.71035004, 0.49119946, 0.46097183]
   bm25:  indices=[389811965] values=[1.6877434]


In [9]:
# Hybrid search with the common parameters written out
from qdrant_client.models import (
    Document, Prefetch, FusionQuery, Fusion, Rrf, RrfQuery,
    Filter, FieldCondition, MatchAny, SearchParams,
)

result = client.query_points(
    collection_name="test",

    # Step 1: candidate searches, each runs on its own
    prefetch=[
        # Dense (semantic) search
        Prefetch(
            query=[0.1, 0.2, 0.3, 0.5],      # query vector (normally from an embedding model)
            using=None,                      # which vector to search; None = the unnamed dense vector
            limit=10,                        # top-k candidates from this search (usually 20-100, > final limit)
            score_threshold=0.5,             # min cosine similarity; closest thing to top-p (a quality cut-off)
            params=SearchParams(hnsw_ef=128, exact=False),  # accuracy vs speed of the vector index
        ),
        # BM25 (keyword) search
        Prefetch(
            query=Document(text="dog", model="qdrant/bm25"),  # query text -> BM25 sparse vector
            using="bm25",                    # search the sparse "bm25" vector
            limit=10,                        # top-k candidates from this search
        ),
    ],

    # Step 2: merge the candidate lists
    query=RrfQuery(rrf=Rrf(
        k=60,                                # RRF constant: score = sum(weight / (k + rank)); 60 is the standard default
        weights=[1.0, 1.0],                  # weight per prefetch, same order as above (e.g. [1, 2] = trust BM25 more)
    )),
    # Simpler alternatives:
    #   query=FusionQuery(fusion=Fusion.RRF)    # RRF with defaults
    #   query=FusionQuery(fusion=Fusion.DBSF)   # merge normalised scores instead of ranks

    # Step 3: what to return
    query_filter=Filter(                     # payload filter (SQL WHERE), applied to every prefetch
        must_not=[FieldCondition(key="text", match=MatchAny(any=["hello", "qdrant"]))],
    ),
    limit=5,                                 # TOP-K: number of final results
    offset=0,                                # skip the first N results (pagination)
    score_threshold=None,                    # min fused score (RRF scores are rank-based, so rarely useful here)
    with_payload=True,                       # return payload: True / False / list of keys like ["text"]
    with_vectors=True,                       # return vectors: True / False / list of names like ["bm25"]
    timeout=10,                              # request timeout in seconds
)
# Note: there is no top-p in search. Top-p / temperature are LLM sampling settings,
# used later when the retrieved chunks are sent to the LLM.

for p in result.points:
    assert isinstance(p.vector, dict)
    print(p.id, round(p.score, 3), p.payload["text"])
    print("   dense:", p.vector[""])
    print("   bm25: ", p.vector["bm25"])

18 0.017 river
   dense: [0.3046928, 0.25708455, 0.19995464, 0.8950351]
   bm25:  indices=[12344922] values=[1.6877434]
6 0.017 dog
   dense: [0.7139128, 0.61696166, 0.2996671, 0.1410198]
   bm25:  indices=[1312749093] values=[1.6877434]
4 0.016 banana
   dense: [0.03452612, 0.23017414, 0.74806595, 0.6214702]
   bm25:  indices=[178115665] values=[1.6877434]
12 0.016 red
   dense: [0.20816116, 0.59846336, 0.26020145, 0.7285641]
   bm25:  indices=[347068675] values=[1.6877434]
14 0.016 blue
   dense: [0.20403671, 0.71035004, 0.49119946, 0.46097183]
   bm25:  indices=[389811965] values=[1.6877434]
